## 3D volume in napari with z-scale and ROIs

loads suite2p results, crops padding artifacts, opens in napari
with proper anisotropic z-scaling and detected ROIs as labels.

In [ ]:
from mbo_utilities import imread
from mbo_utilities.metadata import get_voxel_size, get_param
import numpy as np
from pathlib import Path

data_path = r"X:\lbm\wsnyder\2026-02-13 342191-1\tp00001-47155"
data = imread(data_path)
meta = data.metadata
print(f"shape: {data.shape}, dims: {data.dims}")

### get voxel size for z-scaling

In [ ]:
vs = get_voxel_size(meta)
print(f"voxel size: dx={vs.dx} um, dy={vs.dy} um, dz={vs.dz} um")

z_scale = vs.dz / vs.dx if vs.dx else vs.dz
print(f"z/xy scale ratio: {z_scale:.1f}x")

### crop padding artifacts using ops.npy valid regions

In [ ]:
base = Path(data_path)
yranges = []
xranges = []

for ops_file in sorted(base.glob("zplane*/ops.npy")):
    ops = np.load(ops_file, allow_pickle=True).item()
    yr = ops.get("_pad_yrange") or ops.get("yrange", [0, ops.get("Ly", data.shape[-2])])
    xr = ops.get("_pad_xrange") or ops.get("xrange", [0, ops.get("Lx", data.shape[-1])])
    yranges.append(yr)
    xranges.append(xr)

y0 = max(int(yr[0]) for yr in yranges)
y1 = min(int(yr[1]) for yr in yranges)
x0 = max(int(xr[0]) for xr in xranges)
x1 = min(int(xr[1]) for xr in xranges)
print(f"valid region: y=[{y0}:{y1}], x=[{x0}:{x1}] ({y1-y0}x{x1-x0})")

### build 3D ROI labels from per-plane stat.npy

In [ ]:
nz = data.shape[1] if data.ndim == 4 else 1
Ly_crop = y1 - y0
Lx_crop = x1 - x0

labels_3d = np.zeros((nz, Ly_crop, Lx_crop), dtype=np.uint32)
cell_offset = 0
plane_cell_counts = []

zplane_dirs = sorted(base.glob("zplane*"))
for zi, zdir in enumerate(zplane_dirs):
    stat_file = zdir / "stat.npy"
    if not stat_file.exists():
        plane_cell_counts.append(0)
        continue

    stat = np.load(stat_file, allow_pickle=True)
    count = 0
    for s in stat:
        ypix = s["ypix"]
        xpix = s["xpix"]
        yc = ypix - y0
        xc = xpix - x0
        valid = (yc >= 0) & (yc < Ly_crop) & (xc >= 0) & (xc < Lx_crop)
        if valid.any():
            cell_offset += 1
            count += 1
            labels_3d[zi, yc[valid], xc[valid]] = cell_offset

    plane_cell_counts.append(count)

print(f"total ROIs: {cell_offset}")
for zi, c in enumerate(plane_cell_counts):
    print(f"  zplane {zi+1:02d}: {c} cells")

### load a single timepoint volume for 3D view

In [ ]:
t = 0
vol = np.asarray(data[t, :, y0:y1, x0:x1], dtype=np.float32)
print(f"volume shape: {vol.shape}")

### open in napari

In [ ]:
import napari

viewer = napari.Viewer(ndisplay=3)

scale = (z_scale, 1.0, 1.0)
viewer.add_image(
    vol,
    name="volume",
    scale=scale,
    colormap="gray",
    contrast_limits=[np.percentile(vol, 1), np.percentile(vol, 99.5)],
    rendering="mip",
)

if cell_offset > 0:
    viewer.add_labels(
        labels_3d,
        name=f"ROIs ({cell_offset} cells)",
        scale=scale,
        opacity=0.5,
    )

viewer.camera.angles = (30, -30, 150)
viewer.camera.zoom = 1.5